# Phase 2 Research Gate — one-click runner (Google Colab)

**Automatic:** freezes OKX market data + real historical funding rates →
honest backtest (fees, slippage, funding, pessimistic fills) → bias audits → prints
the reports you paste back to the assistant.

**Usage:**
1. Menu **Runtime → Run all** (or Ctrl/Cmd+F9)
2. Keep this tab open (data freeze is the slow part)
3. Scroll to the **last cell**, copy everything, paste to the assistant

If Colab disconnects mid-download: re-run Cell 3 — already-frozen pairs are skipped.
The notebook auto-switches to the arena work branch if `research/` is not on `main` yet.

In [ ]:
# Cell 1 — fetch the repo (auto-fallback to the work branch when PR not merged yet)
import os
REPO = '/content/ML_ANN_Paper_Bot'
if not os.path.exists(REPO):
    os.system(f'git clone -q https://github.com/ah9mohammad-netizen/ML_ANN_Paper_Bot.git {REPO}')
else:
    os.system(f'git -C {REPO} fetch -q origin --prune')
os.chdir(REPO)
os.system('git checkout -q arena/01a027de-ml-ann-paper-bot || '
          'git checkout -q -b arena/01a027de-ml-ann-paper-bot origin/arena/01a027de-ml-ann-paper-bot || true')
os.system('git reset -q --hard origin/arena/01a027de-ml-ann-paper-bot || true')  # always latest code; parquet data is untracked & survives
if not os.path.exists('research/fetch_data.py'):
    os.system('git checkout -q main')
assert os.path.exists('research/fetch_data.py'), 'research/ missing in repo clone'
print('cwd:', os.getcwd())
print('HEAD:', os.popen('git log --oneline -1').read().strip())

In [ ]:
# Cell 2 — dependencies + OKX connectivity probe
rc = os.system('pip install -q -r requirements-research.txt')
print('pip install rc:', rc)
import requests
ok = False
for host in ['https://aws.okx.com', 'https://www.okx.com', 'https://www.okx.cab', 'https://www.okx.ceo']:
    try:
        t = requests.get(host + '/api/v5/public/time', timeout=10).json()
        if t.get('code') == '0':
            print(f'✅ OKX reachable via {host}'); ok = True; break
    except Exception:
        continue
if not ok:
    raise SystemExit('⛔ OKX unreachable from this runtime — Runtime → Disconnect and delete runtime → Run all')

In [ ]:
# Cell 3 — FREEZE DATA  (slowest part; prints one line per pair; resumes if re-run)
DAYS = 730  # reduce to e.g. 500 for a shorter download
rc = os.system(f'python -m research.fetch_data --days {DAYS}')
print('fetch return code:', rc)
from pathlib import Path
pqs = sorted(Path('research/data/frozen').glob('*.parquet'))
print(f'frozen files: {len(pqs)}')
for f in pqs: print('  ', f.name)
assert rc == 0 and pqs, 'data freeze failed or produced nothing'

In [ ]:
# Cell 4 — RUN THE GATE (full output captured; failures print the real traceback)
import subprocess
p = subprocess.run(['python', '-m', 'research.run_research_gate', '--frozen'],
                   capture_output=True, text=True)
print(p.stdout)
if p.returncode != 0:
    print('STDERR:')
    print(p.stderr)
    print('return code:', p.returncode)
    print('👆 copy EVERYTHING above and paste it to the assistant')
else:
    print('✅ gate complete — proceed to Cell 5')

In [ ]:
# Cell 5 — PRINT THE REPORTS (copy everything below and paste it to the assistant)
from pathlib import Path
out = Path('research/output')
files = sorted(out.glob('REAL_*')) if out.exists() else []
if not files:
    print('⛔ no reports yet. output dir:', sorted(x.name for x in out.glob('*')) if out.exists() else '(missing)')
else:
    for k, f in enumerate(files, 1):
        print('=' * 30, f'REPORT {k} OF {len(files)}: {f.name}', '=' * 30)
        print(f.read_text())
    print('=' * 30, 'END — paste everything above this line', '=' * 30)

In [ ]:
# Optional — download trades.csv + manifest as a zip
os.system('cd research && zip -qr /content/research_output.zip output data/frozen/manifest.json')
try:
    from google.colab import files as gfiles
    gfiles.download('/content/research_output.zip')
except Exception as e:
    print('download skipped (not in Colab):', e)